### Reusable scraping of Barbora

In [ ]:
# TODO I need to cleanup these functions, and add more metadata. Such as how many grams are there. Currently it is inside the name.

In [ ]:
import time, json, re, requests, pandas as pd

In [ ]:
def parse_product_list(html: str):

    marker = "window.b_productList ="

    i = html.find(marker)
    if i == -1:
        return []
    i += len(marker)
    while html[i].isspace():
        i += 1
    products, _ = json.JSONDecoder().raw_decode(html, i)
    
    return products

def normalize_barbora(p):

    return {
        "store": "barbora",
        "id": p["id"],
        "name": p["title"],
        "brand": p.get("brand_name"),
        "price": p["price"],
        "unit_price": p.get("comparative_unit_price"),
        "unit": p.get("comparative_unit"),
        "category": p.get("caregory_name_full_path"),
        "image": p.get("image"),
        "url": "https://barbora.lt/produktai/" + p["Url"],
        "on_promo": p.get("promotion") is not None,
    }

def scrape_category(base_url, max_pages=3, delay=2.0):
    rows = []
    headers = {"User-Agent": "Mozilla/5.0 (course-demo; personal use)"}

    for page in range(1, max_pages + 1):
        r = requests.get(base_url, params={"page": page}, headers=headers, timeout=20)
        r.raise_for_status()
        items = parse_product_list(r.text)
        print(f"page {page}: {len(items)} products")
        if not items:
            break
        rows += [normalize_barbora(p) for p in items]
        time.sleep(delay)
    return rows

rows = scrape_category(
    "https://barbora.lt/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena",
    max_pages=3,
)
df = pd.DataFrame(rows).drop_duplicates("id")
print(len(df))
df.head()

### Explained

In [ ]:
import time, json, re, requests, pandas as pd
# time     -> lets us pause between requests (sleep)
# json     -> turns JSON text into Python lists/dicts
# re       -> regular expressions (not used in this cell, safe to remove)
# requests -> sends HTTP requests, like a browser would
# pandas   -> holds the results in a table (DataFrame)


def parse_product_list(html):
    """Take the raw HTML of a category page and return the list of products
    hidden inside it as a Python list of dictionaries."""

    # The page contains a line like:  window.b_productList = [ {...}, {...} ];
    # This text is the "label" that tells us where the product data begins.
    marker = "window.b_productList ="

    # .find() returns the position of the text, or -1 if it isn't there
    i = html.find(marker)
    if i == -1:
        return []  # no product list on this page (e.g. we went past the last page)

    # Move the position to the end of the marker, so we point right after the "="
    i += len(marker)

    # Skip spaces / newlines until we reach the "[" that opens the list
    while html[i].isspace():
        i += 1

    # raw_decode reads ONE complete JSON value starting at position i, and stops
    # exactly where that value ends. It returns (the_parsed_value, end_position).
    # We only need the value, so the "_" ignores the end position.
    products, _ = json.JSONDecoder().raw_decode(html, i)
    return products


def normalize_barbora(p):
    """Convert one Barbora product dictionary into our own common format.
    Later we'll write the same kind of function for Rimi, so both stores
    produce rows with identical column names."""
    return {
        "store": "barbora",                      # which shop this row came from
        "id": p["id"],                           # [] = field MUST exist, else error
        "name": p["title"],
        "brand": p.get("brand_name"),            # .get = give None if missing, no crash
        "price": p["price"],                     # price of the whole pack, in EUR
        "unit_price": p.get("comparative_unit_price"),  # price per kg or per litre
        "unit": p.get("comparative_unit"),       # which unit the unit_price is for ("kg", "l")
        "category": p.get("category_name_full_path"),   # e.g. "Mėsa, žuvis.../Jautiena"
        "image": p.get("image"),
        # The page gives only the end part of the link (the slug), so we add the
        # shop's address in front to get a full, clickable URL
        "url": "https://barbora.lt/produktai/" + p["Url"],
        # True if the product has a promotion attached, False if promotion is null
        "on_promo": p.get("promotion") is not None,
    }


def scrape_category(base_url, max_pages=3, delay=2.0):
    """Download several pages of one category and return all products as rows."""
    rows = []  # we collect the cleaned products here

    # range(1, max_pages + 1) gives 1, 2, 3 ... up to max_pages
    for page in range(1, max_pages + 1):
        # params={"page": page} adds "?page=2" etc. to the URL automatically.
        # "headers" is defined earlier in your notebook (the User-Agent).
        r = requests.get(base_url, params={"page": page}, headers=headers, timeout=20)

        # If the server answered with an error (403, 500...), stop with a clear message
        r.raise_for_status()

        # Pull the product list out of this page's HTML
        items = parse_product_list(r.text)
        print(f"page {page}: {len(items)} products")

        # An empty list means there are no more products -> stop early
        if not items:
            break

        # Clean every product and append it to our collection
        rows += [normalize_barbora(p) for p in items]

        # Wait before the next request so we don't hammer the server
        time.sleep(delay)

    return rows


# Run it: fetch up to 3 pages of the fresh meat category
rows = scrape_category(
    "https://barbora.lt/mesa-zuvis-ir-kulinarija/sviezia-mesa-ir-paukstiena",
    max_pages=3,
)

# Turn the list of dictionaries into a table.
# drop_duplicates("id") removes any product that appeared twice (same id).
df = pd.DataFrame(rows).drop_duplicates("id")
print(len(df))  # how many unique products we got
df.head()       # show the first 5 rows as a table